# basic knot generation

### creating points from words

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import einops as eo

from geometry_mnist.plotly.templates import register
register()

def plot_pts(pts: np.ndarray, kos: bool = True) -> go.Figure:
    # add the main data
    data = [
        go.Scatter3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            marker=dict(color=np.arange(len(pts)), colorscale="rainbow", size=5),
        )
    ]
    # add a KOS
    if kos:
        data += [
            go.Scatter3d(
                x=[0, 0.5 if ax=="x" else 0], y=[0, 0.5 if ax=="y" else 0], z=[0, 0.5 if ax=="z" else 0],
                mode="lines",
                line=dict(color=col, width=10),
                showlegend=False,
                hoverinfo="skip",
            )
            for ax, col in zip(["x", "y", "z"], ["red", "green", "blue"])
        ]
    # create figure
    fig = go.Figure(
        data=data,
        layout=dict(
            template="mine_dark+mine_3d",
            scene=dict(
                aspectmode="data",
                camera=dict(
                    eye=dict(x=3.0, y=0, z=0.8), # where the camera sits
                    center=dict(x=0, y=0, z=0.0), # the point it looks at
                    up=dict(x=-1, y=0, z=0), # which direction is "up" on screen
                ),
            ),
            height=600
        ),
    )
    return fig

def plot_bpts(bpts: np.ndarray, kos: bool = True) -> go.Figure:
    """ plots a list of point-lists (multiple braids for example or blocks)"""
    
    # add the main data
    data = [
        go.Scatter3d(
            x=b[:, 0],
            y=b[:, 1],
            z=b[:, 2],
        )
        for b in bpts
    ]
    
    # add a KOS
    if kos:
        kos_len = 0.5
        data += [
            go.Scatter3d(
                x=[0, kos_len if ax=="x" else 0], y=[0, kos_len if ax=="y" else 0], z=[0, kos_len if ax=="z" else 0],
                mode="lines",
                line=dict(color=col, width=10),
                showlegend=False,
                hoverinfo="skip",
            )
            for ax, col in zip(["x", "y", "z"], ["red", "green", "blue"])
        ]
    
    # create figure
    fig = go.Figure(
        data=data,
        layout=dict(
            template="mine_dark+mine_3d",
            scene=dict(
                aspectmode="data",
                camera=dict(
                    eye=dict(x=3.0, y=0, z=0.8), # where the camera sits
                    center=dict(x=0, y=0, z=0.0), # the point it looks at
                    up=dict(x=-1, y=0, z=0), # which direction is "up" on screen
                ),
            ),
            height=600
        ),
    )
    return fig

   

DIM = 3

def _sig(x: np.ndarray) -> np.ndarray:
    """ basically a sigmoid to get a smooth transition from y=0 to y=1 over 1x """
    a = 12
    return 1 / (1 + np.exp(-a * (x - 0.5)))

def _hop(x: np.ndarray) -> np.ndarray:
    """ a gaussian bell to hop in z direction for the active crossing braid """
    a = 30
    b = 0.4
    return b * np.exp(-a * (x - 0.5)**2)

def generate_straight(n_points: int) -> np.ndarray:
    """ just the straight line braid within unit square"""
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    return pts
    
def generate_l2r_active(n_points: int) -> np.ndarray:
    """ the L->R transition with a positive zhop. hop can easily be made to go over or under by sign flip. """
    
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    y[:] = _sig(x)
    z[:] = _hop(x)
    return pts
    
def generate_r2l_passive(n_points: int) -> np.ndarray:
    """ the R->L transition without zhop. """
    
    pts = np.zeros(shape=(n_points, DIM))
    x, y, z = pts.T # just a "named view" on the pts column 
    x[:] = np.linspace(0, 1, n_points, endpoint=False)
    y[:] = 1 - _sig(x)
    return pts
    
def generate_newslice(n_points: int, n_braids: int) -> np.ndarray:
    """ helper to generate one op-slice for all braids and fill it with dummy straight lines for each braid. """
    
    # the base line element for one braid at y=z=0 with an extra batch braid dimension for broadcasting
    one_line = generate_straight(n_points)[None, :, :]
    
    # a broadcasting matrix to duplicate the basis line n_braid times and increase y coords by 1 each time
    distributor = np.ones(shape=(n_braids, 1, DIM))
    distributor[:, 0, 1] = np.arange(n_braids) + 1
    
    # elementwise multiplication and broadcasting duplicates the line element with an increasing y coord. temporarily setting the initial y to 1 allows the multiplication to take effect
    one_line[:, :, 1] = 1
    bpts = one_line * distributor
    bpts[:, :, 1] -= 1 # set the line elements to begin at y=0 again

    return bpts



# choosable params
word = [1,-2,1,3,2,2,-4,-3,2,-3,-3,-4] # word must be 1-indexed to hold sign+braid_id. 
n_pts_per_ops = 30

# determined params
n_braids = np.max(np.abs(word)) + 1
n_ops = len(word) 

# generate the braid multi-pointlist
bpts = np.zeros(shape=(n_braids, n_ops*n_pts_per_ops, 3))

# build word as points
for i, op in enumerate(word):
    
    # extract braid_ids and hop_dir; braid indices everywhere else but the word input must be 0-indexed!
    braid_id_active  = abs(op) - 1 # index of the braid that is targeted by the op
    braid_id_passive = abs(op) -1 + 1 # index of the braid that has to move as well to swap with the active one
    hop_dir = np.sign(op) # whether to hop over or under

    # one fresh "op slice"; all lists just get a straight line
    bpts_newslice = generate_newslice(n_points=n_pts_per_ops, n_braids=n_braids)
    
    # then overwrite the two braids that are part of the op with crossing points!
    pts_active = generate_l2r_active(n_points=n_pts_per_ops) # the active braid going from left to right
    pts_passive = generate_r2l_passive(n_points=n_pts_per_ops) # the passive braid (one after) going from right to left
    # adjust y coordinates of points to braid position (they are all generated in the unit square)
    pts_active[:, 1]  += braid_id_active
    pts_passive[:, 1] += (braid_id_active)
    # finally flip sign of z-coords depending on hop direction (to go under or over the passive braid)
    pts_active[:, 2] *= hop_dir 
    # overwrite the straight lines with the two crossings
    bpts_newslice[braid_id_active, :, :] = pts_active
    bpts_newslice[braid_id_passive, :, :] = pts_passive

    # offset the entire new slice to the current "x end position"
    bpts_newslice[:, :, 0] += i
    
    # write the newslice into the main braid-points array
    bpts[:, i*n_pts_per_ops:(i+1)*n_pts_per_ops, :] = bpts_newslice


# now convert these to proper xyz coords (before it was generated basically in pyr)
R = max(1, n_ops / (2*np.pi))

bpts_xyz = np.zeros_like(bpts)


bpts_xyz[:, :, 0] = (R + bpts[:, :, 2]) * np.sin(bpts[:, :, 0]/n_ops * 2*np.pi)
bpts_xyz[:, :, 1] = bpts[:, :, 1]
bpts_xyz[:, : ,2] = (R + bpts[:, :, 2]) * np.cos(bpts[:, :, 0]/n_ops * 2*np.pi)


plot_bpts(bpts_xyz).show()

### Linking the individual operation blocks

In [ ]:
# linking the individual segments!

# create the new singular pointlist
pts_xyz = np.zeros(shape=(n_ops*n_braids*n_pts_per_ops, 3))

# decompose the per-braid lists into it's individual operations and create one list of all the individual op. these sequences are known to be connected points within itself.
block_list = eo.rearrange(bpts_xyz, "braid (block pts) xyz -> (braid block) pts xyz", pts=n_pts_per_ops)

# ----- for each LAST point, find the corresponding closest next FISRT point -----

# extract only the beginning and end points of each block "chain"
first_points = block_list[:, 0, :] # [n_first_pts, 3]
last_points = block_list[:, -1, :] # [n_last_pts, 3]

# batched point differences
diffs = (
    eo.rearrange(last_points, "n_last xyz -> n_last 1 xyz") -
    eo.rearrange(first_points, "n_first xyz -> 1 n_first xyz") 
    
)
squared_dist = eo.reduce(diffs**2, "n_last n_first xyz -> n_last n_first", "sum")

# argmin finds all the links, but they are unordered!
successor_map = np.argmin(squared_dist, axis=1)

# link all the blocks into one list
if len(set(successor_map)) != len(successor_map):
    raise ValueError("ambiguous linking: some block has multiple predecessors")

block_id = 0
order = []
visited = set()

for _ in range(len(successor_map)):
    if block_id in visited:
        raise ValueError(f"knot is made up of disjoint loops!")
    
    visited.add(int(block_id))
    order.append(block_id)
    block_id = successor_map[block_id]
 
pts_xyz = eo.rearrange(block_list[order], "allblocks pts xyz -> (allblocks pts) xyz")

plot_pts(pts_xyz)


In [ ]:
def plot_scrub(pts: np.ndarray) -> go.Figure:
    n = len(pts)

    # ==========================================================================
    # ANIMATION OPTIONS: how a slider step moves to its frame.
    # This is the 2nd element of a step's `args` (= options for Plotly.animate).
    # ==========================================================================
    anim_opts = dict(
        mode="immediate",        # "immediate": drop queued animations, jump now (good for sliders)
                                 # "next" / "afterall": queue behind the running animation
        frame=dict(
            duration=0,          # ms each frame is shown (matters when playing several frames)
            redraw=True,         # full redraw; REQUIRED for 3D (scene traces can't tween)
        ),
        transition=dict(
            duration=0,          # ms of tweening between frames (only works for some 2D traces)
            easing="linear",     # "cubic-in-out", "elastic", ... (only with duration > 0)
        ),
        fromcurrent=False,       # when playing a list of frames: start at the current one
    )

    fig = go.Figure(
        # ======================================================================
        # BASE DATA: the initial state. Frames patch these traces by index.
        # ======================================================================
        data=[
            # trace 0: static curve (no frame touches it)
            go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode="lines+markers",
                         marker_size=4, name="curve"),
            # trace 1: moving red marker (patched by every frame)
            go.Scatter3d(x=pts[:1,0], y=pts[:1,1], z=pts[:1,2], mode="markers",
                         marker=dict(size=8, color="red"), name="current"),
        ],

        # ======================================================================
        # FRAMES: precomputed states. Each one is a PATCH, not a replacement:
        # only the given properties change, everything else (color, size...) stays.
        # ======================================================================
        frames=[
            go.Frame(
                name=str(i),     # ID that slider steps refer to in args[0]
                data=[           # patches, paired by position with `traces`
                    go.Scatter3d(x=pts[i:i+1,0], y=pts[i:i+1,1], z=pts[i:i+1,2]),
                ],
                traces=[1],      # data[k] patches trace traces[k]; same length as data!
                                 # (if left out, data[k] patches trace k)
                layout=dict(),   # optional layout patch per frame, e.g. title_text=f"point {i}"
                                 # (in 3D, scene camera changes also go here)
            )
            for i in range(n)
        ],

        layout=dict(
            template="mine_dark+mine_3d",
            scene_aspectmode="data",

            # ==================================================================
            # SLIDER: a list of steps. Each step calls a Plotly.js method with args.
            # ==================================================================
            sliders=[dict(
                active=0,                    # index of the initially selected step
                steps=[
                    dict(
                        label=str(i),        # display text on the tick only (no linking role)
                        value=str(i),        # value reported in events; defaults to label
                        method="animate",    # Plotly.js function to call:
                                             #   "animate"  -> go to frames (args below)
                                             #   "restyle"  -> args=[{trace props}, [idx]]
                                             #   "relayout" -> args=[{layout props}]
                                             #   "update"   -> args=[{trace}, {layout}, [idx]]
                                             #   "skip"     -> do nothing (pure UI)
                        args=[
                            [str(i)],        # frame name(s) to go to -> must match Frame.name;
                                             # several names = play them in sequence
                            anim_opts,
                        ],
                        execute=True,        # False: only move the slider, don't call `method`
                    )
                    for i in range(n)
                ],

                # current value readout above the slider
                currentvalue=dict(
                    visible=True,
                    prefix="point: ",
                    suffix="",
                    xanchor="left",          # "left" | "center" | "right"
                    offset=10,               # px between readout and slider
                    font=dict(),             # e.g. size=14, color="red"
                ),

                # position & size (paper coordinates, 0..1 = plot area)
                x=0, y=0,                    # anchor point; y=0 + yanchor="top" -> BELOW the plot
                xanchor="left", yanchor="top",
                len=1,                       # length as a fraction of the plot width
                pad=dict(t=20, b=0, l=0, r=0),   # px padding around the slider

                # look
                ticklen=7,                   # tick length in px (0 = no ticks)
                font=dict(),                 # tick label font
                # bgcolor, bordercolor, activebgcolor, tickcolor, ... -> colors

                # animation of the slider HANDLE itself (not the plot)
                transition=dict(duration=150, easing="cubic-in-out"),
            )],
        ),
    )

    return fig

_ = plot_scrub(pts_xyz)

### Normalize the knot somewhat - uniform point resampling

n_resample just needs to be cleeearly large enough so that the link lenght is wayy smaller then the current minimal distances between ropes. shouldn't be too hard, except for possibly really complicated knots

In [ ]:
from scipy.interpolate import make_splprep

n_resample = 512

def get_ropelength(pts: np.ndarray) -> float:
    with_endpoint = np.concat([pts, pts[0, :][None, :]])
    dists = np.linalg.norm(np.diff(with_endpoint, axis=0), ord=2, axis=1)
    length = np.sum(dists)
    return float(length)

spl, u = make_splprep(x=np.vstack([pts_xyz, pts_xyz[:1]]).T, k=3, s=1.2, bc_type="periodic")
pts_dense = spl(np.linspace(0, 1, n_resample, endpoint=False)).T # resampling with arbitrary number of points

# normalize rope length to 1
ropelen = get_ropelength(pts_dense)
print(f"initial ropelength was: {ropelen}")

pts_dense = (pts_dense / ropelen)
pts_dense = pts_dense - pts_dense.mean(0)
ropelen = get_ropelength(pts_dense)
print(f"after rescaling ropelength is: {ropelen}")

plot_pts(pts_dense, kos=False).show()

### relaxation simulator

In [ ]:
# calcuate all the energies! as formulas
import torch

NORM_CONST_CONFINEMENT = (1/00.0127)
NORM_CONST_BENDING = (1/19.7400)
NORM_CONST_FARREPULSION = (1/99.7000)

def nrg_confinement(state: torch.Tensor, link_l: float) -> torch.Tensor:
    """ the further away from the origin all points are the more energy. use square of distances! """
    
    distances_from_origin_squared = (state**2).sum(dim=1)
    e = distances_from_origin_squared.sum()
    
    return e / 2 * link_l # normalizing

def nrg_bending(state: torch.Tensor, link_l: float) -> torch.Tensor:
    """ angle 1 - cos(phi) between each consecutive link"""
    
    v_prev = state - state.roll(shifts=1, dims=0)  # x_i − x_{i−1}: edge from i−1 to i
    v_next = state.roll(shifts=-1, dims=0) - state # x_{i+1} − x_i: edge from i to i+1
    cos_phi = torch.einsum("ij, ij -> i", v_prev, v_next) / (v_prev.norm(dim=1) * v_next.norm(dim=1))
    
    # going via the angle is unnecessary as an arccos corrupts the gradients. this is also square weighted as for small angles the 1-cos(phi) is ~= phi^2 / 2
    e = (1 - cos_phi).sum()
    
    return e / link_l # normalizing

def _generate_repulsion_mask(n_points: int, excl_percentage: float = 0.02) -> torch.Tensor:
    """ mask to exclude the points that are close on the arc (thick diagonal matrix). works for uniform link length """
    
    idx = torch.arange(n_points)
    arc_distances = (idx[:, None] - idx[None, :]).abs()
    arc_distances = torch.minimum(arc_distances, n_points - arc_distances)
    mask = arc_distances > excl_percentage*n_points
    
    return mask
       
def nrg_far_repulsion(state: torch.Tensor, link_l: float) -> torch.Tensor:
    """ some far reaching point repulsion, to separate out entagnled strands, mostly for relaxation"""
    
    distances_squared = ((
        eo.rearrange(state, "n dim -> n 1 dim") - 
        eo.rearrange(state, "n dim -> 1 n dim")
    )**2).sum(dim=2)
    
    # energy is just 1/dist^2 but we mask off the degenerate ii pairs and the ones close on the arc. dont count repulsion between points that are quite close along the arc -> determine whats close as a percentage of the total ropelength (which is just set to be n_sample points!) 
    e = (repulsion_mask * (1 / distances_squared.clip(min=1e-12, max=None))).sum()
    
    return link_l**2 * e # normalizing

def nrg_close_repulsion(state: torch.Tensor) -> torch.Tensor:
    # would be more for simulating actual tubes that cannot intersect and would be used for the dataset generation
    ...

def nrg_total(state: torch.Tensor, link_l: float, k: float, a: float, f: float) -> torch.Tensor:
    """ individual contributions have be normalized to a circle of unit circumference """
    
    total = (
        k * NORM_CONST_CONFINEMENT * nrg_confinement(state, link_l) + 
        a * NORM_CONST_BENDING * nrg_bending(state, link_l) + 
        f * NORM_CONST_FARREPULSION * nrg_far_repulsion(state, link_l)
    )
    
    return total


x = torch.as_tensor(pts_dense, dtype=torch.float64)
npts = x.shape[0]

repulsion_mask = _generate_repulsion_mask(npts)
link_l = 1 / npts


k = 1 # ~ confinement stiffness
a = 1 # ~ bending stiffness (a/Temp/l_total would be even better = persistence length)
f = 1 # ~ repulsion strength (this can also be related to noise?)

e = nrg_total(x, link_l, k, a, f)
print(e)

In [ ]:
def total_ropelength(state: torch.Tensor) -> torch.Tensor:
    with_endpoint = torch.cat([state, state[0, :][None, :]])
    dists = torch.diff(with_endpoint, dim=0).norm(dim=1)
    length = dists.sum()
    return length

def project_links(state: torch.Tensor, link_l: float, iters: int = 10) -> torch.Tensor:
    """PBD: pull each link back to length l0, even/odd edges alternating (N must be even)."""
    N = len(state)
    idx = torch.arange(N)
    for _ in range(iters):
        for start in (0, 1):
            # get even odd point indices
            i = idx[start::2]
            j = (i + 1) % N
            
            d = state[j] - state[i]
            r = d.norm(dim=1, keepdim=True)
            corr = 0.5 * ((r - link_l) / r) * d
            state[i] += corr
            state[j] -= corr
    return state

def get_sobolev_kernel(n_points: int, reach_percentage: float = 0.02) -> torch.Tensor:
    
    # define alpha in terms of the entire chain length (more or less cutoff @37% strength of kernel)
    ell = reach_percentage * n_points
    if ell == 0:
        alpha = 0
    else:
        alpha = 1.0 / (4 * np.sinh(1 / (2*ell))**2)
    
    I = torch.eye(n_points)
    L = -2*I  + I.roll(shifts=1, dims=0) + I.roll(shifts=-1, dims=0)
    A = I - alpha*L
    A_INV = torch.linalg.inv(A)

    return (A_INV).to(torch.float64)

# debug plotting for step ================   
fig, axs = plt.subplots(3, 1, figsize=(14, 14))
axs[0].set_title("gradient-x")
axs[1].set_title("gradient-y")
axs[2].set_title("gradient-z")

plot_i = 0

def step(
    state: torch.Tensor, 
    dt: float, 
    k: float = 1.0, a: float = 1.0, f: float = 1.0, T: float = 0.0, 
    max_disp: float = 0.5 * link_l,
):
    
    # 1. ----- calculate [energy + gradient]
    state = state.detach().requires_grad_(True)
    E = nrg_total(state, link_l=link_l, k=k, a=a, f=f)
    g, *_ = torch.autograd.grad(E, state)

    with torch.no_grad():
        
        # 1.5 ----- precondition gradient (smoothing it out along the chain)
        g = A_INV_POW2@g
        
        # plot gradients debug ===========
        global plot_i
        if (plot_i % 10) == 0:
            axs[0].plot(g[:, 0])
            axs[1].plot(g[:, 1])
            axs[2].plot(g[:, 2])
        plot_i += 1    
        # ==============================
        
        # 2. ----- Brownian step (friction per unit length -> / link_l)
        
        # check if timestep needs to be limited (calculate what the maximum of dx would be and if it's larger than max disp!)
        max_found_disp = (-dt * g / link_l).norm(dim=1).max().item()
        scale = min(1.0, max_disp / max_found_disp)
        dt_corrected = dt * scale
        
        # do the actual update. appyling the "correction" to dt instead of the resulting update step, correctly co-scales the tempeature (which needs to be scaled with sqrt(dt)!)
        dx = -dt_corrected * g / link_l
        if T > 0:
            dx += (2 * T * dt_corrected / link_l)**0.5 * (A_INV_POW1@torch.randn_like(state))
        state = state + dx

        # 4. ----- PBD (+ optional re-center)
        state = state - state.mean(0)
        state = state / total_ropelength(state)
        state = project_links(state, link_l) # makes sense to put this here? as it only can balance not rescale all

    return state.detach(), E.item(), scale


# RUN ======================================================================
x = torch.as_tensor(pts_dense, dtype=torch.float64)
link_l = 1 / x.shape[0]
# the conditioning matrix for smoothing the gradients (pow2) while pow1 is used to scale the noise
A_INV_POW1 = get_sobolev_kernel(n_resample)
A_INV_POW2= torch.linalg.matrix_power(A_INV_POW1, 2) # this is basically like "applying the smoothing kernel twice"

N = 10_000

# persistence length! (how much information is still carried along the chain through temperature jitter
l_persist = 5
k = 1.0
a = 1.0
f = 10.0

T = (1/l_persist) * (a*NORM_CONST_BENDING)
T_vec = np.linspace(T, 0, N) * 0

nrgies = []
states_snapsh = [x]
for i in range(N):
    
    # the actual loop
    x_next, E0, s = step(
        state=x, 
        dt=1e-4,
        k=k, a=a, f=f, T=T_vec[i],
    )
    nrgies.append(E0)
    states_snapsh.append(x_next)
    E1 = nrg_total(x_next, link_l, k=k, a=a, f=f).item()
    x = x_next
    
    # logging and info
    links = (x_next.roll(-1, 0) - x_next).norm(dim=1) / link_l
    print(f"E {i:05d}: {E0:.4f} -> {E1:.4f}   dt_scale={s:.3g}", end="   ")
    print(f"links / l0: min={links.min():.5f}  max={links.max():.5f}")
    
plot_pts(x, kos=False)


In [ ]:
from plotly.subplots import make_subplots

def plot_relaxation(states: list, energies: list | None = None, stride: int = 1) -> go.Figure:
    """ scrub through a list of knot states [n_pts, 3] (tensors or arrays). optional per-state energies get a side plot with a moving marker. """

    idx = list(range(0, len(states), stride))
    if idx[-1] != len(states) - 1:
        idx.append(len(states) - 1) # always include the final state

    def _np(s):
        return s.detach().cpu().numpy() if torch.is_tensor(s) else np.asarray(s)

    # close the loop for drawing
    snaps = [np.vstack([_np(states[i]), _np(states[i])[:1]]) for i in idx]

    # fixed, equal axis ranges over the whole run -> camera doesn't rescale per frame
    half = max(np.abs(s).max() for s in snaps) * 1.05
    axis = dict(range=[-half, half], autorange=False)

    has_e = energies is not None
    fig = make_subplots(
        rows=1, cols=2 if has_e else 1,
        specs=[[{"type": "scene"}, {"type": "xy"}]] if has_e else [[{"type": "scene"}]],
        column_widths=[0.65, 0.35] if has_e else None,
    )

    # trace 0: the knot (patched by every frame)
    p0 = snaps[0]
    fig.add_trace(go.Scatter3d(
        x=p0[:, 0], y=p0[:, 1], z=p0[:, 2], mode="lines",
        line=dict(width=10, color=np.arange(len(p0)), colorscale="rainbow"),
        name="knot", showlegend=False,
    ), row=1, col=1)

    if has_e:
        # trace 1: static energy curve, trace 2: moving marker
        fig.add_trace(go.Scatter(x=np.arange(len(energies)), y=energies, mode="lines", showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=[idx[0]], y=[energies[idx[0]]], mode="markers",
                                 marker=dict(size=10, color="red"), showlegend=False), row=1, col=2)
        fig.update_yaxes(type="log", title_text="energy", row=1, col=2)
        fig.update_xaxes(title_text="step", row=1, col=2)

    frames = []
    for i, p in zip(idx, snaps):
        data = [go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2])]
        traces = [0]
        if has_e:
            j = min(i, len(energies) - 1)
            data.append(go.Scatter(x=[j], y=[energies[j]]))
            traces.append(2)
        frames.append(go.Frame(name=str(i), data=data, traces=traces))
    fig.frames = frames

    anim_opts = dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))
    play_opts = dict(mode="immediate", frame=dict(duration=40, redraw=True), transition=dict(duration=0), fromcurrent=True)

    fig.update_layout(
        template="mine_dark+mine_3d",
        height=650,
        scene=dict(xaxis=axis, yaxis=axis, zaxis=axis, aspectmode="cube"),
        sliders=[dict(
            active=0,
            steps=[dict(label=str(i), method="animate", args=[[str(i)], anim_opts]) for i in idx],
            currentvalue=dict(prefix="step: ", xanchor="left"),
            x=0.1, len=0.9, y=0, yanchor="top", pad=dict(t=30),
        )],
        updatemenus=[dict(
            type="buttons", direction="left",
            x=0, y=0, xanchor="left", yanchor="top", pad=dict(t=30),
            buttons=[
                dict(label="▶", method="animate", args=[None, play_opts]),
                dict(label="⏸", method="animate", args=[[None], anim_opts]),
            ],
        )],
    )
    return fig

# states[i] should be the state BEFORE step i (so it lines up with nrgies[i]):
#   states = [x]; ... in the loop after the step: states.append(x)
plot_relaxation(states_snapsh, energies=nrgies, stride=10)

### cleanup proposal 

In [ ]:
# cleanup proposal claude

from dataclasses import dataclass

@dataclass
class StepControl:
    dt_max: float = 1e-6         # just a ceiling, not a target

    # --- before the step: limits on dt ---
    use_push_link: bool = True   # push: neighbours moving apart
    use_push_cross: bool = True  # push: strand passage
    use_noise_link: bool = True  # noise: neighbours moving apart
    use_noise_cross: bool = True # noise: strand passage

    c_link: float = 0.05         # max change of a link vector per step, in units of l0
    c_cross: float = 0.25        # max displacement per step, as a fraction of d_min
    noise_sigmas: float = 4.0    # "typical max" of the noise = this many std devs

    # --- after the step: verify, reject on failure ---
    use_verify: bool = True
    max_link_error: float = 0.02 # |link/l0 - 1| after PBD
    max_retries: int = 5


def noise_gains(K: torch.Tensor) -> tuple[float, float]:
    """How much the kernel K scales unit noise. K is circulant, so row 0 is representative."""
    gain_abs  = K[0].norm().item()                          # std of (K @ xi)[i]
    gain_link = (K.roll(-1, dims=0) - K)[0].norm().item()   # std of (K @ xi)[i+1] - (K @ xi)[i]
    return gain_abs, gain_link


def min_strand_distance(state: torch.Tensor) -> float:
    """Smallest distance between points that are NOT close along the rope."""
    d2 = ((state[:, None, :] - state[None, :, :]) ** 2).sum(dim=2)
    return d2[repulsion_mask].min().sqrt().item()


def choose_dt(v, d_min, T, ctl: StepControl, gains) -> tuple[float, str]:
    """Return the dt to use, and the name of the limit that decided it (great for logging)."""
    limits = {"dt_max": ctl.dt_max}

    # push: neighbours moving apart
    if ctl.use_push_link:
        dv = (v.roll(-1, dims=0) - v).norm(dim=1).max().item()
        if dv > 0:
            limits["push_link"] = ctl.c_link * link_l / dv

    # push: strand passage
    if ctl.use_push_cross:
        v_max = v.norm(dim=1).max().item()
        if v_max > 0:
            limits["push_cross"] = ctl.c_cross * d_min / v_max

    # noise: std = gain * sqrt(2 T dt / l0), and we want  sigmas * std <= target
    #        -> dt <= (target / (sigmas * gain))^2 * l0 / (2 T)
    if T > 0:
        gain_abs, gain_link = gains
        if ctl.use_noise_link:
            target = ctl.c_link * link_l
            limits["noise_link"] = (target / (ctl.noise_sigmas * gain_link)) ** 2 * link_l / (2 * T)
        if ctl.use_noise_cross:
            target = ctl.c_cross * d_min
            limits["noise_cross"] = (target / (ctl.noise_sigmas * gain_abs)) ** 2 * link_l / (2 * T)

    name = min(limits, key=limits.get)
    return limits[name], name


def verify(x_old, x_new, d_min_old, ctl: StepControl) -> tuple[bool, str]:
    """After the step: did something go wrong anyway?"""
    links = (x_new.roll(-1, dims=0) - x_new).norm(dim=1) / link_l
    if (links - 1).abs().max() > ctl.max_link_error:
        return False, "links"
    # if no point moved more than half the smallest strand distance, no passage was possible
    # (conservative: translation from re-centering also counts)
    if 2 * (x_new - x_old).norm(dim=1).max().item() >= d_min_old:
        return False, "cross"
    return True, ""

# one step !

d_min = min_strand_distance(x)
push = -(A_INV @ g) / link_l
dt, limiter = choose_dt(push, d_min, T, ctl, gains)

for attempt in range(ctl.max_retries):
    dx = dt * push
    if T > 0:
        dx += (2 * T * dt / link_l) ** 0.5 * (K @ torch.randn_like(x))
    x_new = normalize(project_links(x + dx, link_l))     # PBD, center, ropelength

    ok, why = verify(x, x_new, d_min, ctl) if ctl.use_verify else (True, "")
    if ok:
        break
    dt /= 2                                               # retry smaller, with fresh noise

# log: dt, limiter, attempt, E, net displacement (x_new - x).norm(dim=1).max() / link_l